# Notebook 07 — Novelty C: SHAP-Active Feature Drift Detection

## Overview
The base paper uses SHAP purely for **passive explanation** of model
decisions. This notebook upgrades SHAP to an **active anomaly signal**:

1. Compute SHAP values per time window (1,000 sample chunks).
2. Track the **rank order of top features** by mean |SHAP| across windows.
3. Detect windows where the rank order shifts significantly — a
   **pattern shift signal** indicating a change in attack type or traffic.
4. Validate the shift signal against the ground-truth attack-type boundaries
   from Table I of the paper (5 distinct attack types).

**Novel contribution**: SHAP rank dynamics reveal *which attack type* is
active and *when* the attack pattern transitions — information the base
paper does not provide.


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import shap
from scipy.stats import spearmanr, kendalltau
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH    = os.path.join(PROJECT_ROOT, "data", "processed", "ue_attack_labeled_scaled.csv")
MODEL_DIR    = os.path.join(PROJECT_ROOT, "outputs", "models")
FIG_DIR      = os.path.join(PROJECT_ROOT, "outputs", "figures", "nb07_shap_drift")
os.makedirs(FIG_DIR, exist_ok=True)

# Load dataset with timestamps or use index as time proxy
df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape)

X = df.drop(columns=["attack_label"])
y = df["attack_label"].values
feature_names = list(X.columns)

# ── Load model ────────────────────────────────────────────────────────────────
model_path = os.path.join(MODEL_DIR, "xgb_baseline.pkl")
if os.path.exists(model_path):
    model = joblib.load(model_path)
    print("Loaded model:", model_path)
else:
    from sklearn.model_selection import train_test_split
    from imblearn.over_sampling import SMOTE
    import xgboost as xgb
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    X_tr_r, y_tr_r = SMOTE(random_state=42).fit_resample(X_tr, y_tr)
    model = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
    model.fit(X_tr_r, y_tr_r)
    joblib.dump(model, model_path)
    print("Retrained model.")

explainer = shap.TreeExplainer(model)
print("SHAP explainer ready.")


## Step 1 — SHAP Values per Time Window

We compute mean absolute SHAP importance for each 1,000-sample window.
This gives a ranking vector of features for each window.


In [ ]:
WINDOW_SIZE = 1000   # larger windows for stable SHAP estimates
TOP_K       = 8      # track top-K features

n_windows = len(df) // WINDOW_SIZE

shap_importance_matrix = []   # (n_windows, n_features)
attack_type_per_window = []

for i in range(n_windows):
    s  = i * WINDOW_SIZE
    e  = s + WINDOW_SIZE
    Xi = X.iloc[s:e]
    yi = y[s:e]

    sv = explainer.shap_values(Xi)
    if isinstance(sv, list):
        sv = sv[1]          # binary: take positive class

    mean_abs = np.abs(sv).mean(axis=0)
    shap_importance_matrix.append(mean_abs)

    # Assign dominant attack label (majority in window)
    attack_frac = yi.mean()
    attack_type_per_window.append("attack" if attack_frac > 0.5 else "benign")

    if i % 10 == 0:
        print(f"Window {i+1}/{n_windows} processed...")

shap_importance_matrix = np.array(shap_importance_matrix)
print("SHAP importance matrix shape:", shap_importance_matrix.shape)


## Step 2 — Feature Rank Tracking

Convert each window's importance vector to a rank vector. Then track
the rank of the global top-K features across all windows.


In [ ]:
# Global top-K features by overall mean |SHAP|
global_mean_shap = shap_importance_matrix.mean(axis=0)
global_top_k_idx = np.argsort(global_mean_shap)[::-1][:TOP_K]
global_top_k_names = [feature_names[i] for i in global_top_k_idx]

print("Global top features by mean |SHAP|:")
for rank, (idx, name) in enumerate(zip(global_top_k_idx, global_top_k_names)):
    print(f"  Rank {rank+1:2d}: {name:40s} (mean |SHAP| = {global_mean_shap[idx]:.4f})")

# Per-window rank matrix
rank_matrix = np.argsort(np.argsort(-shap_importance_matrix, axis=1), axis=1)
# rank_matrix[w, f] = rank of feature f in window w (0 = most important)

# Extract ranks of top-K features
top_k_ranks = rank_matrix[:, global_top_k_idx]  # (n_windows, TOP_K)


In [ ]:
# ── Plot top-K feature ranks over time ────────────────────────────────────────
cmap = plt.cm.tab10
fig, ax = plt.subplots(figsize=(15, 6))

for k in range(TOP_K):
    ax.plot(range(n_windows), top_k_ranks[:, k],
            label=global_top_k_names[k],
            color=cmap(k / TOP_K), lw=1.4, alpha=0.85)

# Shade attack windows
for w_idx, at in enumerate(attack_type_per_window):
    if at == "attack":
        ax.axvspan(w_idx - 0.5, w_idx + 0.5, alpha=0.12, color="crimson")

attack_patch = mpatches.Patch(color="crimson", alpha=0.3, label="Attack window")
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles + [attack_patch], labels + ["Attack window"],
          bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

ax.invert_yaxis()   # rank 0 at top
ax.set_xlabel("Window index")
ax.set_ylabel("Feature rank (lower = more important)")
ax.set_title(f"Top-{TOP_K} Feature Rank Dynamics Over Time (SHAP-based)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "shap_rank_dynamics.png"), dpi=150)
plt.show()
print("Saved rank dynamics plot.")


## Step 3 — Rank Shift Detection

We compute a **rank shift score** between consecutive windows using the
Kendall tau distance (rank correlation). A low tau (or large 1-tau) 
means the ranking changed significantly — a pattern shift signal.


In [ ]:
rank_shift_scores = []

for i in range(1, n_windows):
    tau, _ = kendalltau(rank_matrix[i-1], rank_matrix[i])
    rank_shift_scores.append(1.0 - tau)   # 0 = identical, 2 = reversed

rank_shift_scores = np.array(rank_shift_scores)

# Detect shifts above mean + 1.5*std
shift_mean = rank_shift_scores.mean()
shift_std  = rank_shift_scores.std()
shift_threshold = shift_mean + 1.5 * shift_std
flagged_windows = np.where(rank_shift_scores > shift_threshold)[0] + 1  # +1 for offset

print(f"Rank shift — mean  : {shift_mean:.4f}")
print(f"Rank shift — std   : {shift_std:.4f}")
print(f"Shift threshold    : {shift_threshold:.4f}")
print(f"Flagged windows    : {len(flagged_windows)}")


In [ ]:
# ── Plot rank shift signal ────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.plot(range(1, n_windows), rank_shift_scores, color="royalblue", lw=1.2,
          label="Rank shift score")
ax1.axhline(shift_threshold, color="crimson", ls="--", lw=1.2,
             label=f"Threshold (μ+1.5σ = {shift_threshold:.3f})")
ax1.scatter(flagged_windows[flagged_windows < n_windows],
            rank_shift_scores[flagged_windows[flagged_windows < n_windows] - 1],
            color="red", zorder=5, s=30, label="Pattern shift flagged")

# Shade attack windows
for w_idx, at in enumerate(attack_type_per_window):
    if at == "attack":
        ax1.axvspan(w_idx - 0.5, w_idx + 0.5, alpha=0.10, color="orange")

orange_patch = mpatches.Patch(color="orange", alpha=0.3, label="Attack window")
handles, labels = ax1.get_legend_handles_labels()
ax1.legend(handles + [orange_patch], labels + ["Attack window"], loc="upper right")

ax1.set_xlabel("Window index")
ax1.set_ylabel("Rank shift score (1 − Kendall τ)")
ax1.set_title("SHAP-Based Feature Rank Shift: Pattern Change Detection")
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "shap_rank_shift_signal.png"), dpi=150)
plt.show()
print("Saved rank shift signal plot.")


## Step 4 — Validate Against Attack-Type Boundaries

From Table I of the paper, attack transitions occur at known time points.
Since we use a scaled dataset (no original timestamps), we identify
attack-type boundaries as windows where the ground-truth label switches
from benign to attack or between attacks.


In [ ]:
# ── Find true transition windows ─────────────────────────────────────────────
# A transition is a window where the dominant label changes from the previous window
transition_windows = []
for i in range(1, n_windows):
    if attack_type_per_window[i] != attack_type_per_window[i-1]:
        transition_windows.append(i)

print(f"True transition windows (label changes): {transition_windows}")

# ── Evaluate: do flagged shift windows coincide with transitions? ─────────────
detection_radius = 3   # flag within ±3 windows of a transition

true_positives  = 0
false_positives = 0

for fw in flagged_windows:
    matched = any(abs(fw - tw) <= detection_radius for tw in transition_windows)
    if matched:
        true_positives += 1
    else:
        false_positives += 1

false_negatives = sum(
    not any(abs(fw - tw) <= detection_radius for fw in flagged_windows)
    for tw in transition_windows
)

precision = true_positives / (true_positives + false_positives + 1e-10)
recall    = true_positives / (true_positives + false_negatives + len(transition_windows) + 1e-10)

print(f"\nPattern Shift Detector Performance (radius={detection_radius}):")
print(f"  True Positives  : {true_positives}")
print(f"  False Positives : {false_positives}")
print(f"  True transitions: {len(transition_windows)}")
print(f"  Precision       : {precision:.4f}")
print(f"  Recall          : {recall:.4f}")


## Step 5 — SHAP Beeswarm by Attack Type (Publication Figure)

Reproduce and extend Fig. 4(d) from the paper by showing SHAP beeswarms
for benign-dominant vs attack-dominant windows separately.


In [ ]:
from sklearn.model_selection import train_test_split
# Sample 500 benign and 500 attack for beeswarm
sample_benign = X[y == 0].sample(min(500, (y == 0).sum()), random_state=42)
sample_attack = X[y == 1].sample(min(500, (y == 1).sum()), random_state=42)

sv_benign = explainer.shap_values(sample_benign)
sv_attack = explainer.shap_values(sample_attack)

if isinstance(sv_benign, list):
    sv_benign = sv_benign[1]
    sv_attack = sv_attack[1]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

plt.sca(axes[0])
shap.summary_plot(sv_benign, sample_benign, show=False, max_display=12)
axes[0].set_title("SHAP (Benign Windows)", fontsize=12)

plt.sca(axes[1])
shap.summary_plot(sv_attack, sample_attack, show=False, max_display=12)
axes[1].set_title("SHAP (Attack Windows)", fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "shap_beeswarm_benign_vs_attack.png"), dpi=150,
            bbox_inches="tight")
plt.show()
print("Saved SHAP beeswarm comparison.")


In [ ]:
# ── Save shift signal for ablation ───────────────────────────────────────────
import json

results = {
    "n_flagged_shifts"   : int(len(flagged_windows)),
    "n_true_transitions" : int(len(transition_windows)),
    "shift_precision"    : float(precision),
    "shift_recall"       : float(recall),
    "shift_threshold"    : float(shift_threshold),
    "mean_rank_shift"    : float(shift_mean),
    "std_rank_shift"     : float(shift_std),
}

save_path = os.path.join(PROJECT_ROOT, "outputs", "nb07_shap_drift_results.json")
with open(save_path, "w") as f:
    json.dump(results, f, indent=2)
print("Saved SHAP drift results.")
for k, v in results.items():
    print(f"  {k:35s}: {v}")


## Summary — Notebook 07

SHAP values, when tracked over time as a rank signal rather than used for
single-inference explanation, reveal:
- **When** the feature importance structure changes (pattern shift signal).
- **Which features** become dominant during different attack types.

This is a novel contribution not present in the base paper, which uses SHAP
only for post-hoc static explanation on the full dataset.

Results feed into Notebook 08 (ablation study).
